In [ ]:
# =============================================================================
# 预处理检查 Notebook (Preprocess Check)
# 用于检查数据预处理是否成功
# =============================================================================

# 运行此notebook前，请先运行 run_preprocess.py 生成预处理数据
# 例如: python run_preprocess.py --data_path /path/to/data.csv --output_dir ./preprocessed_data

import os
import json
import pickle
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 配置
PREPROCESS_OUTPUT_DIR = "./preprocessed_data"  # 修改为你的预处理输出目录

print("=" * 80)
print("📋 预处理检查 Notebook (Preprocess Check)")
print("=" * 80)
print(f"\n预处理输出目录: {PREPROCESS_OUTPUT_DIR}")

## 1. 加载预处理元信息

首先加载 `preprocess_meta.json` 查看预处理的整体情况

In [ ]:
# 加载预处理元信息
meta_path = os.path.join(PREPROCESS_OUTPUT_DIR, "preprocess_meta.json")

if not os.path.exists(meta_path):
    print("❌ 错误: 预处理元信息文件不存在!")
    print(f"   请先运行: python run_preprocess.py --data_path <your_data.csv> --output_dir {PREPROCESS_OUTPUT_DIR}")
    raise FileNotFoundError(f"找不到文件: {meta_path}")

with open(meta_path, 'r', encoding='utf-8') as f:
    preprocess_meta = json.load(f)

print("✅ 预处理元信息加载成功!")
print(f"\n📅 预处理时间: {preprocess_meta['run_time']}")
print(f"⏱️ 耗时: {preprocess_meta['duration_seconds']:.2f} 秒")
print(f"✓ 成功状态: {preprocess_meta['success']}")
print(f"\n📁 配置:")
for key, val in preprocess_meta['config'].items():
    print(f"   - {key}: {val}")

## 2. 检查 fillna 情况

检查哪些字段被 fillna 处理过，这是重要的数据质量检查点

In [ ]:
# 检查 fillna 情况
print("=" * 80)
print("🔍 检查 fillna 情况 (Check fillna Status)")
print("=" * 80)

edge_feat_meta = preprocess_meta['steps'].get('edge_features', {})
fillna_info = edge_feat_meta.get('fillna_info', {})

if not fillna_info:
    print("\n✅ 没有发现 fillna 记录")
else:
    print("\n⚠️ 以下字段被 fillna 处理:")
    
    # 数值特征的fillna
    if 'numerical' in fillna_info:
        print("\n📊 数值特征 (Numerical Features):")
        for col_name, info in fillna_info['numerical'].items():
            print(f"   - {col_name}:")
            print(f"     填充数量: {info['fillna_count']:,}")
            print(f"     填充比例: {info['fillna_percent']:.2f}%")
            print(f"     填充值: {info['fillna_value']}")
    
    # 时间特征的fillna
    if 'time' in fillna_info:
        print("\n⏰ 时间特征 (Time Features):")
        for col_name, info in fillna_info['time'].items():
            if 'parse_failed_count' in info:
                print(f"   - {col_name}:")
                print(f"     解析失败数量: {info['parse_failed_count']:,}")
                print(f"     解析失败比例: {info['parse_failed_percent']:.2f}%")

## 3. 检查 time_diff 特征是否成功构造

time_diff = tds_dt - txn_dt，表示从事件发生到入库的延迟时间

In [ ]:
# 检查 time_diff 特征
print("=" * 80)
print("⏱️ 检查 time_diff 特征 (Check time_diff Feature)")
print("=" * 80)

time_diff_info = edge_feat_meta.get('time_diff_info', {})

if not time_diff_info:
    print("\n❌ 未找到 time_diff 信息")
elif time_diff_info.get('computed', False):
    print("\n✅ time_diff 特征构造成功!")
    
    stats = time_diff_info.get('stats', {})
    print(f"\n📊 时间差统计 (单位: 秒):")
    print(f"   - 有效值数量: {stats.get('valid_count', 'N/A'):,}")
    print(f"   - 无效值数量: {stats.get('invalid_count', 'N/A'):,}")
    print(f"   - 平均延迟: {stats.get('mean_seconds', 'N/A'):.2f} 秒")
    print(f"   - 中位延迟: {stats.get('median_seconds', 'N/A'):.2f} 秒")
    print(f"   - 最小值: {stats.get('min_seconds', 'N/A'):.2f} 秒")
    print(f"   - 最大值: {stats.get('max_seconds', 'N/A'):.2f} 秒")
    print(f"   - 标准差: {stats.get('std_seconds', 'N/A'):.2f} 秒")
    
    negative_count = stats.get('negative_count', 0)
    if negative_count > 0:
        print(f"\n⚠️ 警告: 发现 {negative_count:,} 个负值 (入库时间早于事件时间)")
    
    # fillna 情况
    fillna = time_diff_info.get('fillna_info', {})
    if fillna:
        print(f"\n📝 fillna 情况:")
        print(f"   - 填充数量: {fillna.get('fillna_count', 'N/A'):,}")
        print(f"   - 填充比例: {fillna.get('fillna_percent', 'N/A'):.2f}%")
        print(f"   - 填充方式: {fillna.get('fillna_value', 'N/A')}")
else:
    print("\n❌ time_diff 特征构造失败!")
    if 'error' in time_diff_info:
        print(f"   错误信息: {time_diff_info['error']}")

## 4. 检查边特征

查看边特征的构建情况和特征名称

In [ ]:
# 检查边特征
print("=" * 80)
print("📊 检查边特征 (Check Edge Features)")
print("=" * 80)

edge_features_info = edge_feat_meta.get('edge_features', {})

if edge_features_info:
    print(f"\n✅ 边特征构建成功!")
    print(f"   Shape: {edge_features_info.get('shape', 'N/A')}")
    print(f"   特征数量: {edge_features_info.get('num_features', 'N/A')}")
    
    print(f"\n📋 边特征列表:")
    for i, name in enumerate(edge_features_info.get('feature_names', [])):
        print(f"   {i}: {name}")
else:
    print("\n❌ 边特征信息缺失")

## 5. 检查节点特征

查看节点特征的构建情况和特征名称

In [ ]:
# 检查节点特征
print("=" * 80)
print("🔷 检查节点特征 (Check Node Features)")
print("=" * 80)

node_features_info = preprocess_meta['steps'].get('node_features', {})

if node_features_info:
    print(f"\n✅ 节点特征构建成功!")
    print(f"   Shape: {node_features_info.get('shape', 'N/A')}")
    
    print(f"\n📋 节点特征列表:")
    for i, name in enumerate(node_features_info.get('feature_names', [])):
        print(f"   {i}: {name}")
else:
    print("\n❌ 节点特征信息缺失")

## 6. 检查图数据

加载实际的图数据文件并验证

In [ ]:
# 加载并检查图数据
print("=" * 80)
print("📂 检查图数据文件 (Check Graph Data Files)")
print("=" * 80)

# 检查文件是否存在
files_to_check = [
    "graph_data.pt",
    "node_features.pt",
    "edge_features.pt",
    "edge_index.pt",
    "node_mapping.pkl",
    "statistics.json",
    "preprocess_meta.json"
]

print("\n📁 文件检查:")
all_exist = True
for filename in files_to_check:
    filepath = os.path.join(PREPROCESS_OUTPUT_DIR, filename)
    exists = os.path.exists(filepath)
    status = "✅" if exists else "❌"
    size = os.path.getsize(filepath) / 1024 if exists else 0
    print(f"   {status} {filename}: {size:.2f} KB" if exists else f"   {status} {filename}: 不存在")
    if not exists:
        all_exist = False

if all_exist:
    print("\n✅ 所有预处理文件都已生成!")
else:
    print("\n⚠️ 部分文件缺失，请检查预处理是否完整运行")

In [ ]:
# 加载实际的图数据
print("=" * 80)
print("📊 加载图数据 (Load Graph Data)")
print("=" * 80)

try:
    # 加载图数据
    graph_path = os.path.join(PREPROCESS_OUTPUT_DIR, "graph_data.pt")
    data = torch.load(graph_path)
    
    print(f"\n✅ 图数据加载成功!")
    print(f"\n📊 图数据属性:")
    print(f"   - 节点数 (num_nodes): {data.num_nodes:,}")
    print(f"   - 边数-含自环 (edge_index): {data.edge_index.shape[1]:,}")
    print(f"   - 节点特征维度 (x): {data.x.shape}")
    
    if hasattr(data, 'original_edge_index'):
        print(f"   - 边数-原始 (original_edge_index): {data.original_edge_index.shape[1]:,}")
    
    if hasattr(data, 'edge_attr') and data.edge_attr is not None:
        print(f"   - 边特征维度 (edge_attr): {data.edge_attr.shape}")
    
    if hasattr(data, 'edge_weight') and data.edge_weight is not None:
        print(f"   - 边权重 (edge_weight): {data.edge_weight.shape}")
        
except Exception as e:
    print(f"\n❌ 图数据加载失败: {e}")

## 7. 数据质量检查

检查特征数据中是否有 NaN、Inf 等异常值

In [ ]:
# 数据质量检查
print("=" * 80)
print("🔍 数据质量检查 (Data Quality Check)")
print("=" * 80)

def check_tensor_quality(tensor, name):
    """检查张量的数据质量"""
    arr = tensor.numpy()
    nan_count = np.isnan(arr).sum()
    inf_count = np.isinf(arr).sum()
    zero_count = (arr == 0).sum()
    total = arr.size
    
    print(f"\n   {name}:")
    print(f"     Shape: {arr.shape}")
    print(f"     NaN 数量: {nan_count} ({nan_count/total*100:.2f}%)")
    print(f"     Inf 数量: {inf_count} ({inf_count/total*100:.2f}%)")
    print(f"     零值数量: {zero_count} ({zero_count/total*100:.2f}%)")
    print(f"     最小值: {np.nanmin(arr):.4f}")
    print(f"     最大值: {np.nanmax(arr):.4f}")
    print(f"     均值: {np.nanmean(arr):.4f}")
    
    if nan_count > 0 or inf_count > 0:
        return False
    return True

try:
    print("\n📊 节点特征质量:")
    node_ok = check_tensor_quality(data.x, "x (节点特征)")
    
    if hasattr(data, 'edge_attr') and data.edge_attr is not None:
        print("\n📊 边特征质量:")
        edge_ok = check_tensor_quality(data.edge_attr, "edge_attr (边特征)")
    else:
        edge_ok = True
    
    if node_ok and edge_ok:
        print("\n✅ 数据质量检查通过!")
    else:
        print("\n⚠️ 数据质量存在问题，请检查预处理流程")
        
except Exception as e:
    print(f"\n❌ 数据质量检查失败: {e}")

## 8. 统计信息检查

查看图的统计信息

In [ ]:
# 加载统计信息
print("=" * 80)
print("📈 统计信息 (Statistics)")
print("=" * 80)

stats_path = os.path.join(PREPROCESS_OUTPUT_DIR, "statistics.json")

try:
    with open(stats_path, 'r', encoding='utf-8') as f:
        statistics = json.load(f)
    
    # 基本统计
    if 'basic' in statistics:
        basic = statistics['basic']
        print(f"\n📊 基本统计:")
        print(f"   - 节点数: {basic.get('num_nodes', 'N/A'):,}")
        print(f"   - 边数(原始): {basic.get('num_edges_original', 'N/A'):,}")
        print(f"   - 边数(含自环): {basic.get('num_edges_with_loops', 'N/A'):,}")
        print(f"   - 图密度: {basic.get('graph_density', 'N/A'):.10f}")
    
    # 度数统计
    if 'degree' in statistics and 'total_degree' in statistics['degree']:
        deg = statistics['degree']['total_degree']
        print(f"\n📈 度数统计:")
        print(f"   - 平均度数: {deg.get('mean', 'N/A'):.2f}")
        print(f"   - 最大度数: {deg.get('max', 'N/A'):.0f}")
        print(f"   - 最小度数: {deg.get('min', 'N/A'):.0f}")
    
    # 金额统计
    if 'amount_distribution' in statistics:
        print(f"\n💰 金额分布:")
        for key, val in statistics['amount_distribution'].items():
            if 'error' not in val:
                print(f"   - {key}: 均值={val.get('mean', 0):,.2f}, 中位数={val.get('median', 0):,.2f}")
    
    print("\n✅ 统计信息加载成功!")
    
except Exception as e:
    print(f"\n❌ 统计信息加载失败: {e}")

## 9. 可视化检查

绘制一些基本的分布图来直观检查数据

In [ ]:
# 可视化检查
print("=" * 80)
print("📊 可视化检查 (Visualization Check)")
print("=" * 80)

try:
    from torch_geometric.utils import degree
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 1. 节点度数分布
    ax = axes[0, 0]
    edge_index = data.original_edge_index if hasattr(data, 'original_edge_index') else data.edge_index
    degrees = degree(edge_index[0], num_nodes=data.num_nodes) + degree(edge_index[1], num_nodes=data.num_nodes)
    degrees = degrees.numpy()
    ax.hist(degrees, bins=50, edgecolor='black', alpha=0.7)
    ax.set_xlabel('Node Degree')
    ax.set_ylabel('Count')
    ax.set_title('Node Degree Distribution')
    ax.set_yscale('log')
    
    # 2. 节点特征分布（前4个特征）
    ax = axes[0, 1]
    x_np = data.x.numpy()
    for i in range(min(4, x_np.shape[1])):
        ax.hist(x_np[:, i], bins=50, alpha=0.5, label=f'Feature {i}')
    ax.set_xlabel('Feature Value')
    ax.set_ylabel('Count')
    ax.set_title('Node Feature Distribution (First 4 Features)')
    ax.legend()
    
    # 3. 边特征分布（如果存在）
    ax = axes[1, 0]
    if hasattr(data, 'edge_attr') and data.edge_attr is not None:
        edge_np = data.edge_attr.numpy()
        for i in range(min(4, edge_np.shape[1])):
            ax.hist(edge_np[:, i], bins=50, alpha=0.5, label=f'Edge Feature {i}')
        ax.set_xlabel('Feature Value')
        ax.set_ylabel('Count')
        ax.set_title('Edge Feature Distribution (First 4 Features)')
        ax.legend()
    else:
        ax.text(0.5, 0.5, 'No Edge Features', ha='center', va='center', fontsize=14)
        ax.set_title('Edge Feature Distribution')
    
    # 4. 度数统计条形图
    ax = axes[1, 1]
    if 'degree' in statistics:
        deg_types = ['in_degree', 'out_degree', 'total_degree']
        means = [statistics['degree'].get(t, {}).get('mean', 0) for t in deg_types]
        maxs = [statistics['degree'].get(t, {}).get('max', 0) for t in deg_types]
        
        x_pos = np.arange(len(deg_types))
        width = 0.35
        ax.bar(x_pos - width/2, means, width, label='Mean', color='steelblue')
        ax.bar(x_pos + width/2, [m/100 for m in maxs], width, label='Max/100', color='coral')
        ax.set_xticks(x_pos)
        ax.set_xticklabels(['In-Degree', 'Out-Degree', 'Total'])
        ax.set_ylabel('Value')
        ax.set_title('Degree Statistics')
        ax.legend()
    
    plt.tight_layout()
    plt.show()
    
    print("\n✅ 可视化完成!")
    
except Exception as e:
    print(f"\n❌ 可视化失败: {e}")

## 10. 检查汇总

汇总所有检查项的结果

In [ ]:
# 检查汇总
print("=" * 80)
print("📋 预处理检查汇总 (Preprocess Check Summary)")
print("=" * 80)

checks = {
    "预处理成功完成": preprocess_meta.get('success', False),
    "time_diff 特征构造": time_diff_info.get('computed', False),
    "边特征生成": len(edge_features_info.get('feature_names', [])) > 0,
    "节点特征生成": len(node_features_info.get('feature_names', [])) > 0,
    "图数据文件存在": all_exist,
    "节点特征无NaN": 'data' in dir() and not torch.isnan(data.x).any().item(),
    "节点特征无Inf": 'data' in dir() and not torch.isinf(data.x).any().item()
}

print("\n检查项目               状态")
print("-" * 40)
all_passed = True
for check_name, passed in checks.items():
    status = "✅ 通过" if passed else "❌ 失败"
    print(f"{check_name:20s}  {status}")
    if not passed:
        all_passed = False

print("\n" + "=" * 80)
if all_passed:
    print("🎉 所有检查通过! 预处理数据可用于 main 部分的模型训练")
    print("\n下一步: 运行 main 部分")
    print("   cd ../")
    print("   python run_main.py --preprocessed_dir ./preprocess/preprocessed_data")
else:
    print("⚠️ 部分检查未通过，请检查预处理流程")
print("=" * 80)